<img src="logoucm.png" style="height: 200px">

<font size="5"><center>***Gestión de riesgos financieros (parte III).***</center></font>

<font size="4"><center>***Master en Ingeniería Matemática.***</center></font>

<font size="5"><center><span style="color:blue">***Nombre y Apellidos: _______________________***</span></center></font>

---
<font size="5"><center><span style="color:darkred">***Prueba de Evaluación — Riesgo de Mercado en Tiempos de Guerra Comercial (2025–2026)***</span></center></font>

**Instrucciones generales:**
- Rellena tu nombre completo en la celda de código siguiente antes de empezar.
- El notebook debe ejecutarse limpiamente de principio a fin (`Kernel → Restart & Run All`) antes de la entrega.
- Para cada bloque hay celdas de código (que debes completar) y cuestiones teóricas (que debes responder en la celda Markdown indicada).
- Los datos están en el fichero `datos_examen_GRF.pkl` adjunto.
- **Puntuación orientativa:** Bloque 1 (25 pts), Bloque 2 (25 pts), Bloque 3 (25 pts), Bloque 4 (25 pts).


In [ ]:
nombreyapellidos = ""
if nombreyapellidos == "":
    print("Rellena tu nombre completo antes de continuar!")
else:
    print("Gracias:", nombreyapellidos)

<img src="Contexto_problema_2026.png" width="1200" alt="Texto descriptivo">

---
| Bloque | Contenido | Herramientas principales |
|--------|-----------|-------------------------|
| 1 | Análisis individual de los activos | Retornos log, hechos estilizados, GARCH(1,1), VaR/ES individual |
| 2 | Valoración y riesgo de la opción Put | Black-Scholes-Merton, Delta, Gamma, aproximación Delta-Gamma |
| 3 | Cartera de acciones (parte lineal) | Matriz Σ, VaR paramétrico, Monte Carlo con Cholesky |
| 4 | Cartera completa con derivado | VaR histórico (full revaluation), impacto de la Put, síntesis |


---
## Librerías necesarias


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm, t as t_dist
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns
from arch import arch_model
import pickle
import warnings; warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 4)
np.random.seed(42)


---
## Carga de datos


In [ ]:
with open("datos_examen_GRF_2026.pkl", "rb") as f:
    data = pickle.load(f)

prices = data["Adj Close"]

TICKERS    = ["NVDA", "BMW.DE", "CAT"]
NOMBRES    = {"NVDA": "NVIDIA", "BMW.DE": "BMW Group", "CAT": "Caterpillar"}
TICKER_OPC = "NVDA"

INI_PRE  = "2024-10-01"
FIN_PRE  = "2025-03-31"
INI_POST = "2025-04-02"
FIN_POST = "2025-06-30"

sub_pre  = prices.loc[INI_PRE:FIN_PRE,   TICKERS].dropna()
sub_post = prices.loc[INI_POST:FIN_POST,  TICKERS].dropna()

rets_pre  = np.log(sub_pre  / sub_pre.shift(1)).dropna()
rets_post = np.log(sub_post / sub_post.shift(1)).dropna()

prec0 = sub_pre.iloc[-1]   # precios en t=0 (último día del periodo pre)

print(f"Periodo pre  (estimacion) : {sub_pre.index[0].date()}  a  {sub_pre.index[-1].date()}  ({len(rets_pre)} sesiones)")
print(f"Periodo post (estres)     : {sub_post.index[0].date()}  a  {sub_post.index[-1].date()}  ({len(rets_post)} sesiones)")
print()
print("Precios de cierre en t=0:")
for t in TICKERS:
    print(f"  {NOMBRES[t]:<14}: {prec0[t]:.2f} USD/EUR")


---
<font size="6"><center><span style="color:blue">***Bloque 1 — Análisis Individual de los Activos***</span></center></font>

Modelizamos la distribución marginal de los retornos de cada acción y calculamos el VaR y el ES individuales bajo distintos supuestos distribucionales. Trabajamos siempre con el **periodo pre** (oct 2024 – mar 2025) como ventana de estimación.

Los retornos logarítmicos diarios se definen como:
$$R_t = \ln\!\left(\frac{S_t}{S_{t-1}}\right)$$


### 1.1 — Hechos estilizados de los retornos


In [ ]:
# --- 1.1 Estadísticos descriptivos ---
# Para cada activo en rets_pre calcula:
#   · Media diaria
#   · Desviacion tipica diaria
#   · Volatilidad anualizada (× sqrt(252))
#   · Asimetria (skewness)
#   · Curtosis en exceso (excess kurtosis)
#   · p-valor del test de Jarque-Bera
# Presenta los resultados en un DataFrame bien formateado.

# TU CODIGO AQUI


In [ ]:
# --- 1.2 Histograma + KDE vs normal teorica (un subplot por activo) ---
# Para cada activo muestra histograma, KDE empirica y normal teorica
# con la misma media y desviacion tipica.

# TU CODIGO AQUI


**Cuestion 1.1** Los tres activos muestran exceso de curtosis positivo. ¿Que implica esto para el uso del VaR parametrico bajo normalidad? ¿En que direccion sesga el error (sobreestima o infravalora el riesgo real)? Justifica tu respuesta en el contexto de los shocks arancelarios de 2025.

*Escribe tu respuesta aqui:*


### 1.2 — Modelizacion GARCH(1,1) de la volatilidad


No asumiremos volatilidad constante. Estimamos un modelo **GARCH(1,1)** para cada activo:

$$\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2$$

Parametros clave:
- **Persistencia:** $\alpha + \beta$ (cuanto mas proximo a 1, mas lenta la reversion al largo plazo)
- **Varianza incondicional:** $\bar{\sigma}^2 = \dfrac{\omega}{1-\alpha-\beta}$

Usa la libreria `arch` con `vol="GARCH"`, `p=1`, `q=1`, distribucion normal. Multiplica los retornos por 100 antes de estimar.


In [ ]:
# --- 1.3 Estimacion GARCH(1,1) para cada activo ---
# Almacena los resultados en: garch_results = {ticker: result_object}
# Para cada activo imprime: omega, alpha, beta, persistencia y vol LP anualizada.

garch_results = {}

# TU CODIGO AQUI


In [ ]:
# --- 1.4 Volatilidad condicional GARCH vs retornos ---
# Para cada activo, panel con dos subplots apilados:
#   Superior: retornos diarios del periodo pre
#   Inferior: volatilidad condicional anualizada del modelo GARCH
# Aniade una nota indicando el Liberation Day (2025-04-02).

# TU CODIGO AQUI


**Cuestion 1.2** Interpreta los parametros estimados $\hat{\alpha}$ y $\hat{\beta}$ para NVIDIA. ¿Es la persistencia alta o baja? ¿Que implica para la velocidad de reversion a la volatilidad de largo plazo? ¿Como esperas que se refleje en los retornos del periodo de estres post-Liberation Day?

*Escribe tu respuesta aqui:*


### 1.3 — VaR y ES individuales: normalidad vs t de Student vs Cornish-Fisher


Calculamos VaR y ES al **95% y 99%** para horizonte de **1 dia** bajo tres supuestos:

**Supuesto 1 — Normalidad:**
$$\text{VaR}_{\alpha}^{\text{N}} = -(\mu + z_{\alpha}\,\sigma), \qquad \text{ES}_{\alpha}^{\text{N}} = -\mu + \sigma\,\frac{\phi(z_{\alpha})}{\alpha}$$

**Supuesto 2 — t de Student** (estima $\nu$ con `scipy.stats.t.fit`):
$$\text{VaR}_{\alpha}^{t} = -(\mu + t_{\alpha,\nu}\,\hat{s}), \qquad \text{ES}_{\alpha}^{t} = -\mu + \hat{s}\,\frac{f_t(t_{\alpha,\nu})}{\alpha}\cdot\frac{\nu + t_{\alpha,\nu}^2}{\nu - 1}$$

**Correccion de Cornish-Fisher** *(formula dada, no es necesario derivarla)*:

La correccion ajusta el cuantil normal incorporando la asimetria ($S$) y la curtosis en exceso ($K$) empiricas:
$$z_{\alpha}^{\text{CF}} = z_{\alpha} + \frac{z_{\alpha}^2 - 1}{6}\,S + \frac{z_{\alpha}^3 - 3z_{\alpha}}{24}\,K - \frac{2z_{\alpha}^3 - 5z_{\alpha}}{36}\,S^2$$
$$\text{VaR}_{\alpha}^{\text{CF}} = -(\mu + z_{\alpha}^{\text{CF}}\,\sigma)$$
donde $z_{\alpha} = \Phi^{-1}(\alpha)$, $S$ es la asimetria muestral y $K$ la curtosis en exceso muestral.


In [ ]:
# --- Funcion Cornish-Fisher (YA IMPLEMENTADA, no modificar) ---

def var_cornish_fisher(returns, alpha=0.01):
    """
    VaR de Cornish-Fisher al nivel (1-alpha).
    Devuelve el VaR como numero positivo (perdida en fraccion del valor).
    """
    mu    = returns.mean()
    sigma = returns.std(ddof=1)
    S     = stats.skew(returns)
    K     = stats.kurtosis(returns)   # curtosis en exceso
    z     = norm.ppf(alpha)
    z_cf  = (z
             + (z**2 - 1) / 6 * S
             + (z**3 - 3*z) / 24 * K
             - (2*z**3 - 5*z) / 36 * S**2)
    return -(mu + z_cf * sigma)

print("Funcion Cornish-Fisher cargada correctamente.")


In [ ]:
# --- 1.5 Calculo de VaR y ES individuales ---
# Para cada activo en rets_pre, calcula y presenta en un DataFrame:
#   · VaR 95% y 99% bajo: Normal, t-Student y Cornish-Fisher
#   · ES  95% y 99% bajo: Normal y t-Student
# Expresa los resultados en % (multiplica por 100).
#
# Pista ES t-Student:
#   nu, loc, scale = t_dist.fit(rets, floc=0)
#   t_q = t_dist.ppf(alpha, df=nu)
#   ES  = -(loc + scale * t_dist.pdf(t_q, df=nu) / alpha * (nu + t_q**2) / (nu - 1))

# TU CODIGO AQUI


In [ ]:
# --- 1.6 Backtesting visual (periodo post-Liberation Day) ---
# Representa para NVIDIA los retornos del periodo post (rets_post["NVDA"])
# junto con las lineas horizontales del VaR99 Normal y VaR99 t-Student
# estimados con rets_pre. Resalta con puntos de distinto color los dias
# en que el retorno supera (en perdida) cada uno de los dos VaR.
# Indica en el titulo el numero de violaciones para cada modelo.

# TU CODIGO AQUI


**Cuestion 1.3** ¿Cuantas veces se supera el VaR99 Normal en el periodo de estres? ¿Y el VaR99 t-Student? ¿Cual infravalora mas el riesgo real durante los shocks arancelarios? Relaciona tu respuesta con los hechos estilizados del apartado 1.1.

*Escribe tu respuesta aqui:*


---
<font size="6"><center><span style="color:blue">***Bloque 2 — Valoracion y Riesgo de la Opcion Put***</span></center></font>

El fondo tiene contratada una posicion larga en **opciones Put europeas sobre NVIDIA** como cobertura ante caidas del sector tecnologico. Valoramos la opcion, calculamos sus griegas y medimos su riesgo de forma aislada mediante la aproximacion Delta-Gamma.


### 2.1 — Parametros de la opcion y valoracion Black-Scholes-Merton


In [ ]:
# --- Parametros de la opcion Put europea sobre NVDA ---
S0_nvda = prec0["NVDA"]           # precio NVDA en t=0
K_put   = round(S0_nvda * 0.95)   # strike al 95% del precio actual (ligeramente OTM)
r_c     = np.log(1.045)           # tipo libre de riesgo continuo anual (~4.5%)
d_c     = 0.0
T_put   = 90 / 252                # 90 sesiones hasta vencimiento
N_puts  = 100                     # contratos en cartera

# Volatilidad: extrae la volatilidad condicional GARCH de NVDA en t=0.
# Pista: garch_results["NVDA"].conditional_volatility[-1] / 100
#        (la libreria arch trabaja con retornos en %, hay que volver a fraccion)

# TU CODIGO AQUI: asigna el valor correcto a sigma_nvda
sigma_nvda = None

print(f"Precio NVDA en t=0 (S0) : {S0_nvda:.2f} USD")
print(f"Strike de la Put  (K)   : {K_put:.2f} USD")
print(f"Tiempo a vencimiento    : {T_put:.4f} anyos  ({int(T_put*252)} sesiones)")
print(f"Volatilidad GARCH NVDA  : {sigma_nvda*100:.2f}%  (anualizada)")
print(f"Tipo libre de riesgo    : {np.exp(r_c)-1:.2%}  (anual efectivo)")


La formula de **Black-Scholes-Merton** para una Put europea es:

$$P_0 = K\,e^{-r_c T}\,\Phi(-d_2) - S_0\,\Phi(-d_1)$$

donde:
$$d_1 = \frac{\ln(S_0/K) + (r_c + \sigma^2/2)\,T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$


In [ ]:
# --- 2.1 Implementa la funcion bsm_put ---

def bsm_put(S, K, T, r, sigma, d=0.0):
    """
    Precio BSM de una Put europea.
    S, K, T, r, sigma, d: precio, strike, tiempo (anyos), tipo rf continuo,
                           volatilidad anualizada, dividendo continuo.
    """
    # TU CODIGO AQUI
    pass


P0      = bsm_put(S0_nvda, K_put, T_put, r_c, sigma_nvda, d_c)
V0_puts = N_puts * P0

print(f"Prima Put P0                   : {P0:.4f} USD/accion")
print(f"Valor posicion ({N_puts} contratos): {V0_puts:.2f} USD")
print(f"Apalancamiento S0/P0           : {S0_nvda/P0:.1f}x")


### 2.2 — Las Griegas: Delta y Gamma


Para una **Put europea** bajo BSM:

$$\Delta_{\text{put}} = \Phi(d_1) - 1 \qquad \text{(siempre negativo)}$$

$$\Gamma = \frac{\phi(d_1)}{S_0\,\sigma\sqrt{T}} \qquad \text{(igual para Call y Put)}$$

donde $\phi$ es la densidad de la normal estandar.


In [ ]:
# --- 2.2 Implementa delta_put y gamma_opt ---

def delta_put(S, K, T, r, sigma, d=0.0):
    """Delta de una Put europea: dP/dS"""
    # TU CODIGO AQUI
    pass


def gamma_opt(S, K, T, r, sigma, d=0.0):
    """Gamma de una opcion europea (igual para Call y Put): d2P/dS2"""
    # TU CODIGO AQUI
    pass


Delta0 = delta_put(S0_nvda, K_put, T_put, r_c, sigma_nvda, d_c)
Gamma0 = gamma_opt(S0_nvda, K_put, T_put, r_c, sigma_nvda, d_c)

print(f"Delta de la Put : {Delta0:.6f}")
print(f"Gamma de la Put : {Gamma0:.6f}")
print()
print(f"  Si NVDA sube 1 USD, la prima de la Put cambia en {Delta0:.4f} USD")
print(f"  Si NVDA sube 1 USD, el Delta de la Put cambia en {Gamma0:.6f}")


**Cuestion 2.1** La opcion Put esta ligeramente out-of-the-money ($S_0 > K$). ¿Que implica sobre el signo y el valor absoluto del Delta? ¿Por que el apalancamiento ($S_0/P_0$) es tan elevado? ¿En que direccion beneficia al fondo este apalancamiento si se produce un shock bajista como el de Liberation Day?

*Escribe tu respuesta aqui:*


### 2.3 — VaR de la opcion aislada: aproximacion Delta-Gamma


Para medir el riesgo de la posicion en Puts de forma aislada usamos la **aproximacion cuadratica Delta-Gamma**:

$$\Delta P \approx \Delta_{\text{put}}\,\Delta S + \frac{1}{2}\,\Gamma\,(\Delta S)^2$$

El shock de precio del subyacente $\Delta S$ se obtiene del VaR99 t-Student de NVDA calculado en el Bloque 1:

$$\Delta S^{\text{VaR}} = S_0 \cdot \left(e^{-\text{VaR}_{99}^{\text{NVDA}}} - 1\right)$$

(el signo negativo refleja que es un movimiento adverso —bajada— para la posicion larga en acciones).


In [ ]:
# --- 2.3 VaR de la Put mediante aproximacion Delta-Gamma ---
#
# (a) Recupera el VaR99 t-Student de NVDA del Bloque 1.
#     Transformalo en shock de precio: dS = S0_nvda * (exp(-VaR99) - 1)
#
# (b) Calcula el cambio en la prima bajo tres metodos:
#     · Aproximacion Delta:      dP_delta = Delta0 * dS
#     · Aproximacion Delta-Gamma: dP_dg   = Delta0 * dS + 0.5 * Gamma0 * dS**2
#     · Valoracion BSM completa: dP_bsm   = bsm_put(S0+dS, K_put, T_put-1/252, r_c, sigma_nvda) - P0
#
# (c) Presenta el VaR de la posicion (N_puts contratos) en los tres casos.
#     Indica cual se aproxima mas al valor BSM completo.

# TU CODIGO AQUI


**Cuestion 2.2** La aproximacion Delta-Gamma se acerca mas al BSM completo que la Delta simple. ¿Por que mejora la Gamma la estimacion? ¿En que tipo de escenario —movimiento pequeño, moderado o salto extremo como un anuncio arancelario sorpresa— esperarias que la aproximacion Delta-Gamma todavia cometiese un error apreciable? Justificalo.

*Escribe tu respuesta aqui:*


---
<font size="6"><center><span style="color:blue">***Bloque 3 — Cartera de Acciones (Parte Lineal)***</span></center></font>

Agrupamos las tres acciones en una cartera con pesos $\mathbf{w} = [w_1, w_2, w_3]^\top$, $\sum_i w_i = 1$. Calculamos el VaR de la cartera **sin derivado** bajo dos metodologias: parametrica (varianza-covarianza) y simulacion Monte Carlo con descomposicion de Cholesky.


### 3.1 — Composicion y valoracion de la cartera de acciones


In [ ]:
# --- Posiciones en acciones ---
n_acciones = {"NVDA": 10, "BMW.DE": 20, "CAT": 15}

# Calcula:
#   · Valor de cada posicion = n_i * S0_i
#   · Valor total de la cartera V0_acc
#   · Pesos w_i = (n_i * S0_i) / V0_acc
# Presenta una tabla: activo | n acciones | precio t=0 | valor posicion | peso (%)
# Guarda el vector de pesos como array numpy: w = np.array([w_NVDA, w_BMW, w_CAT])

# TU CODIGO AQUI


### 3.2 — VaR parametrico (varianza-covarianza)


Bajo normalidad multivariante, el VaR de la cartera es:

$$\text{VaR}_{\alpha}(\text{cartera}) = z_{1-\alpha}\,\sqrt{\mathbf{w}^\top\,\hat{\Sigma}\,\mathbf{w}}\;\times\; V_0$$

donde $\hat{\Sigma}$ es la matriz de covarianzas diaria estimada con los retornos del periodo pre y $V_0$ es el valor total de la cartera de acciones.


In [ ]:
# --- 3.1 Estimacion de Sigma y VaR parametrico ---
#
# (a) Estima la matriz de covarianzas diaria con rets_pre.
# (b) Calcula la volatilidad diaria de la cartera: sigma_p = sqrt(w @ Sigma @ w)
# (c) Calcula VaR 95% y 99% parametrico en % y en USD (× V0_acc).
# (d) Muestra la matriz de correlaciones como heatmap (sns.heatmap).

# TU CODIGO AQUI


**Cuestion 3.1** Observa la matriz de correlaciones. ¿Que par de activos presenta la correlacion mas alta? ¿Y la mas baja? ¿Que consecuencias tiene una correlacion elevada entre activos del bloque expuesto para la diversificacion y para el VaR parametrico?

*Escribe tu respuesta aqui:*


### 3.3 — VaR por simulacion Monte Carlo (descomposicion de Cholesky)


Simulamos $N = 50\,000$ escenarios de retornos conjuntos asumiendo:
$$\mathbf{R} \sim \mathcal{N}(\mathbf{0},\,\Sigma)$$

La simulacion se realiza mediante **descomposicion de Cholesky**: si $\Sigma = L L^\top$, entonces:
$$\mathbf{R}^{(i)} = L\,\mathbf{z}^{(i)}, \qquad \mathbf{z}^{(i)} \sim \mathcal{N}(\mathbf{0}, I_3)$$
reproduce la estructura de covarianzas. El rendimiento de la cartera en cada escenario es $r_p^{(i)} = \mathbf{w}^\top \mathbf{R}^{(i)}$.


In [ ]:
# --- 3.2 Simulacion Monte Carlo con Cholesky ---
N_SIM = 50_000

# Pasos:
#   1. L = np.linalg.cholesky(Sigma)
#   2. z ~ N(0, I) de dimension (N_SIM, 3)
#   3. R_sim = z @ L.T   →   shape (N_SIM, 3)
#   4. r_p = R_sim @ w
#   5. VaR 95% y 99% como cuantil empirico de r_p
#   6. ES  95% y 99% como media de las perdidas mas alla del VaR
# Presenta los resultados en % y en USD.

# TU CODIGO AQUI


In [ ]:
# --- 3.3 Visualizacion comparativa ---
# Panel con dos subplots:
#   Izquierda: distribucion de rendimientos MC (histograma + KDE) y
#              retornos historicos del periodo post superpuestos.
#              Marca el VaR99 de ambos con lineas verticales.
#   Derecha:   QQ-plot de los rendimientos MC vs normal teorica.

# TU CODIGO AQUI


**Cuestion 3.2** Compara el VaR 99% parametrico con el VaR 99% Monte Carlo. ¿Son similares? ¿Por que deberia ser asi? Ahora comparalos con los retornos reales del periodo post. ¿Que metodologia se queda mas corta a la hora de capturar las perdidas extremas? ¿A que lo atribuyes?

*Escribe tu respuesta aqui:*


---
<font size="6"><center><span style="color:blue">***Bloque 4 — Cartera Completa con el Derivado***</span></center></font>

Incorporamos la posicion larga en **100 Puts sobre NVDA** a la cartera de acciones. Medimos el impacto del derivado sobre el riesgo total usando el **VaR historico completo** (revaluacion BSM escenario a escenario) y analizamos como cambia el perfil de riesgo con y sin cobertura.


### 4.1 — Valor total de la cartera completa en t=0


In [ ]:
# --- 4.1 Desglose del valor de la cartera completa ---
# La cartera completa = cartera de acciones (Bloque 3) + 100 Puts sobre NVDA
# Calcula y presenta:
#   · Valor acciones = sum(n_i * S0_i)
#   · Valor puts     = N_puts * P0
#   · Valor total V0 = Valor acciones + Valor puts
#   · Peso del derivado en la cartera (%)

# TU CODIGO AQUI


### 4.2 — VaR historico completo (full revaluation)


El VaR historico proyecta los precios de mañana usando los **retornos observados** del periodo pre, revalua la cartera entera (acciones + Put con BSM) y obtiene la distribucion de P&L.

Para cada escenario historico $t = 1, \ldots, T$:
1. Precios de acciones mañana: $S_{t+1}^{(i)} = S_0^{(i)} \cdot e^{R_t^{(i)}}$
2. Precio de la Put mañana: $P_{t+1} = P_{\text{BSM}}\!\left(S_{t+1}^{\text{NVDA}},\, K,\, T_{\text{put}} - \tfrac{1}{252},\, \sigma,\, r\right)$
3. Valor de la cartera mañana: $V_{t+1} = \sum_i n_i\, S_{t+1}^{(i)} + N_{\text{puts}} \cdot P_{t+1}$
4. P&L absoluto: $\text{P\&L}_t = V_{t+1} - V_0$


In [ ]:
# --- 4.2 VaR historico: solo acciones vs cartera completa ---

T_put_1d = T_put - 1/252   # tiempo a vencimiento reducido en 1 dia habil

# Para cada retorno historico en rets_pre:
#   (a) Proyecta los precios de las 3 acciones manana
#   (b) Valora la Put: bsm_put(S_nvda_manana, K_put, T_put_1d, r_c, sigma_nvda)
#   (c) P&L cartera solo acciones  = sum_i n_i * (S_i_mañana - prec0[i])
#   (d) P&L cartera completa       = P&L_acc + N_puts * (P_manana - P0)
#
# Calcula VaR 95% y 99% y ES 95% y 99% para ambas carteras.
# Presenta una tabla comparativa clara.

# TU CODIGO AQUI


In [ ]:
# --- 4.3 Visualizacion: distribuciones de P&L ---
# Panel con dos subplots:
#   Izquierda: histograma + KDE del P&L de solo acciones vs cartera completa
#              (superpuestos). Marca el VaR99 de cada una con lineas verticales.
#   Derecha:   P&L de la posicion en Puts (N_puts contratos) vs precio de NVDA
#              (perfil de payoff del derivado).

# TU CODIGO AQUI


**Cuestion 4.1** La posicion larga en Puts, ¿reduce o amplia el VaR de la cartera respecto a la de solo acciones? ¿Por que? ¿Ocurriria lo mismo si la posicion fuera corta en Puts (vendedor de la opcion)? Relaciona la respuesta con el perfil de payoff asimetrico del derivado.

*Escribe tu respuesta aqui:*


### 4.3 — Cuantas Puts optimizan la cobertura?


In [ ]:
# --- 4.4 Sensibilidad del VaR99 al numero de contratos de Put ---
# Repite el calculo del VaR99 historico de la cartera completa
# variando N_puts en el rango [0, 300] con pasos de 10.
# Representa el VaR99 (en USD) en funcion del numero de contratos.
# Identifica y marca en el grafico el numero de contratos que minimiza el VaR99.

# TU CODIGO AQUI


**Cuestion 4.2** El VaR minimo no se alcanza ni con 0 ni con el maximo de contratos, sino en un punto intermedio. ¿Que concepto de la teoria de carteras explica este fenomeno? ¿Por que a partir de cierto numero de contratos añadir mas Puts comienza a aumentar el riesgo de la cartera?

*Escribe tu respuesta aqui:*


### 4.4 — Tabla comparativa final y sintesis


In [ ]:
# --- 4.5 Tabla resumen de todos los VaR calculados a lo largo de la prueba ---
#
# Construye un DataFrame con el siguiente esquema (completa todos los valores):
#
#  Activo/Cartera        | Metodologia           | VaR 95%  | VaR 99%  | ES 99%
#  ----------------------|-----------------------|----------|----------|--------
#  NVDA individual       | Normal                |          |          |
#  NVDA individual       | t-Student             |          |          |
#  NVDA individual       | Cornish-Fisher        |          |          |
#  BMW individual        | t-Student             |          |          |
#  CAT individual        | t-Student             |          |          |
#  Put NVDA aislada      | Delta-Gamma           |          |          |
#  Cartera acciones      | Parametrico Normal    |          |          |
#  Cartera acciones      | Monte Carlo Normal    |          |          |
#  Cartera completa      | Historico BSM full    |          |          |

# TU CODIGO AQUI


<img src="Sintesis_final_2026.png" width="1200" alt="Texto descriptivo">

---
## Fin de la prueba

Antes de entregar:
1. Comprueba que tu nombre esta relleno en la primera celda de codigo.
2. Ejecuta `Kernel → Restart & Run All` y verifica que no hay errores.
3. Guarda el fichero con el nombre: `GRF_Examen_ApellidoNombre.ipynb`

---
*Asignatura: Introduccion a la Gestion de Riesgos Financieros (Parte III) — Riesgo de Mercado*  
*Master en Ingenieria Matematica — UCM*
